Brought to You by: https://github.com/400lbhacker/RAZORWIRE-LIVE/blob/main/README.md

In [ ]:
# @title 1. 📦 Installation & Setup { vertical-output: true }

# 1. Install radare2 (The way that works in Colab)
!git clone --depth 1 https://github.com/radareorg/radare2
!./radare2/sys/install.sh
# Verification shell command
!r2 -v

# 2. Install Python dependencies with specific versions
!pip install gradio pygments
!pip install transformers>=4.37.0 accelerate torch

# 3. Set up environment
import os
import getpass
import subprocess

# 4. CAPTURE VERSION SAFELY
r2_version_output = !r2 -v
r2_v = r2_version_output[0] if r2_version_output else "Not Found"

# Verify installation
print("\n✅ Setup complete!")
print(f"Radare2 version: {r2_v}")

In [ ]:
# @title 2. 🚀 LIVE Binary Analysis with Radare2 + Forensic Classifier + AI (Qwen 0.5B) { vertical-output: true }

import gradio as gr
import subprocess
import json
import re
import os
import tempfile
import time
import io

# ============================================================================
# AI Chat Assistant with Qwen 0.5B (Tokenless) - FIXED VERSION
# ============================================================================
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

class LocalLLMAssistant:
    def __init__(self, model_name="Qwen/Qwen2.5-0.5B"):
        self.model_name = model_name
        self.pipeline = None
        self.model_loaded = False
        self.loading_error = None
        self.context = ""
        self.tokenizer = None
        self.model = None

    def load_model(self):
        """Load model with CPU optimization"""
        try:
            print(f"🔄 Loading {self.model_name}...")

            # Load tokenizer first
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.model_name,
                trust_remote_code=True,
                padding_side="left"
            )

            # Set pad token if not set
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            # Load model with memory efficient settings
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                torch_dtype=torch.float32,  # Use float32 for CPU
                low_cpu_mem_usage=True,
                trust_remote_code=True
            )

            self.model_loaded = True
            print(f"✅ Qwen 0.5B loaded successfully on CPU")
            return True

        except Exception as e:
            self.loading_error = str(e)
            print(f"❌ Failed to load model: {e}")
            return False

    def set_context(self, ctx):
        self.context = ctx

    def query(self, question, context=""):
        if not self.model_loaded:
            if not self.load_model():
                return f"❌ Model load failed: {self.loading_error}"

        # Use provided context or stored context
        ctx = context if context else self.context

        # Truncate context to prevent token overflow
        if len(ctx) > 2000:
            ctx = ctx[:2000] + "...\n[CONTEXT TRUNCATED]"

        # Simple prompt format for Qwen
        prompt = f"""You are a malware analyst. Based on this forensic data, answer the question.

FORENSIC DATA:
{ctx}

QUESTION: {question}

ANSWER (be concise, reference specific evidence):"""

        try:
            # Tokenize input
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)

            # Generate
            with torch.no_grad():
                outputs = self.model.generate(
                    inputs.input_ids,
                    max_new_tokens=150,
                    temperature=0.3,
                    do_sample=True,
                    top_p=0.95,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )

            # Decode
            response = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

            if response:
                return response.strip()
            else:
                return "I couldn't generate a response based on the forensic data."

        except Exception as e:
            return f"⚠️ LLM error: {str(e)}"

# ============================================================================
# Helper Functions for String Validation
# ============================================================================
def looks_like_real_path(s):
    """Validate if a string looks like a real Windows path"""
    if len(s) < 6:
        return False
    if not re.match(r'^[A-Za-z]:\\', s):
        return False
    if not re.search(r'[a-zA-Z]{2,}', s):
        return False
    alnum_ratio = sum(c.isalnum() for c in s) / len(s)
    if alnum_ratio < 0.5:
        return False
    return True

def looks_like_readable_string(s):
    """Filter out garbage strings"""
    if len(s) < 5 or len(s) > 200:
        return False
    if all(c in '0123456789abcdefABCDEF' for c in s):
        return False
    special_ratio = sum(not c.isalnum() for c in s) / len(s)
    if special_ratio > 0.4:
        return False
    if not re.search(r'[a-zA-Z]{3,}', s):
        return False
    return True

# File type detection
class FileTypeDetector:
    EXECUTABLE_EXTS = {
        '.exe', '.dll', '.sys', '.drv', '.cpl', '.scr', '.ocx', '.ax', '.acm',
        '.efi', '.com', '.bin', '.elf', '.so', '.o', '.ko', '.out', '.app',
        '.dylib', '.bundle', '.framework', '.deb', '.rpm', '.apk', '.dex',
        '.jar', '.class', '.war', '.ear', '.jnilib', '.node', '.wasm'
    }

    WINDOWS_EXTS = {'.exe', '.dll', '.sys', '.drv', '.cpl', '.scr', '.ocx', '.ax', '.acm'}

    @classmethod
    def detect(cls, filename):
        ext = os.path.splitext(filename)[1].lower()

        file_type = {
            'ext': ext,
            'is_executable': ext in cls.EXECUTABLE_EXTS,
            'is_windows': ext in cls.WINDOWS_EXTS,
            'is_dll': ext == '.dll',
            'is_exe': ext == '.exe',
            'r2_flags': ['-e bin.relocs.apply=true'],
            'analysis_commands': []
        }

        if file_type['is_dll']:
            file_type['description'] = "Windows DLL (Dynamic Link Library)"
            file_type['analysis_commands'] = ['aaaa', 'iE', 'ii', 'iS', 'iz', 'afl']
        elif file_type['is_exe']:
            file_type['description'] = "Windows Executable"
            file_type['analysis_commands'] = ['aaaa', 'afl', 'ii', 'iS', 'iz', 'ie']
        elif file_type['is_windows']:
            file_type['description'] = f"Windows Binary ({ext})"
            file_type['analysis_commands'] = ['aaaa', 'afl', 'ii', 'iS', 'iz']
        else:
            file_type['description'] = f"Binary ({ext})"
            file_type['analysis_commands'] = ['aaaa', 'afl', 'iz', 'iS']
            file_type['r2_flags'] = []

        return file_type

# ============================================================================
# Forensic Buffer Classifier with Evidence Collection
# ============================================================================
class ForensicBufferClassifier:
    def __init__(self, analyzer):
        self.analyzer = analyzer
        self.buffer = []
        self.max_buffer_size = 100000

        # Signal scoring engine
        self.weights = {
            "network": 3, "crypto": 4, "key_material": 5, "file_ops": 2,
            "process_ops": 4, "registry_ops": 2, "persistence": 4,
            "anti_debug": 4, "ransomware": 6, "banking": 6,
            "licensing": 5, "cracking": 7, "anti_tamper": 5
        }
        self.signal_counts = {k: 0 for k in self.weights}
        self.correlated_behaviors = []
        self.iocs = {"urls": [], "ips": [], "domains": [], "paths": [], "emails": []}
        self.key_material = []
        self.protectors_detected = []

        # RAW evidence storage
        self.raw_function_hits = []
        self.raw_import_hits = []
        self.raw_string_hits = []
        self.raw_export_hits = []
        self.raw_section_hits = []

        # Pattern sets
        self.network_patterns = [
            'socket', 'connect', 'recv', 'send', 'http', 'https', 'ftp', 'dns',
            'url', 'uri', 'winhttp', 'wininet', 'internetopen', 'ws2_32',
            'tcp', 'udp', 'ipv4', 'ipv6', 'download', 'upload', 'getaddrinfo'
        ]

        self.crypto_patterns = [
            'aes', 'rsa', 'rc4', 'chacha', 'salsa20', 'blowfish', 'twofish',
            'crypt', 'encrypt', 'decrypt', 'bcrypt', 'ncrypt', 'pbkdf',
            'sha', 'md5', 'hmac', 'hash', 'digest', 'cipher', 'blockchain'
        ]

        self.key_patterns = [
            'key', 'iv', 'pem', 'base64', 'secret', 'token', 'password',
            'credential', 'auth', 'bearer', 'apikey', 'api_key', 'private',
            'cert', 'certificate', 'x509'
        ]

        self.file_patterns = [
            'createfile', 'writefile', 'readfile', 'deletefile', 'movefile',
            'copyfile', 'getfileattributes', 'setfileattributes', 'findfirstfile',
            'findnextfile', 'closehandle', 'readdirectory'
        ]

        self.process_patterns = [
            'createprocess', 'openprocess', 'virtualalloc', 'virtualprotect',
            'writeprocessmemory', 'readprocessmemory', 'createremotethread',
            'resumethread', 'suspendthread', 'terminateprocess', 'winexec',
            'shellexecute', 'system'
        ]

        self.registry_patterns = [
            'regopenkey', 'regcreatekey', 'regqueryvalue', 'regsetvalue',
            'regdeletekey', 'regdeletevalue', 'regclosekey', 'shgetvalue'
        ]

        self.persistence_patterns = [
            'run', 'runonce', 'startup', 'service', 'scheduler', 'task',
            'currentversion\\run', 'winlogon', 'explorer', 'shell'
        ]

        self.anti_debug_patterns = [
            'isdebuggerpresent', 'checkremotedebugger', 'ntglobalflag',
            'outputdebugstring', 'peb', 'beingdebugged', 'int3', 'int 2d',
            'rdtsc', 'timing', 'anti', 'debugbreak', 'exception', 'trap'
        ]

        self.licensing_patterns = [
            'license', 'serial', 'activation', 'trial', 'expire', 'expiration',
            'product key', 'registration', 'register', 'genuine', 'validation'
        ]

        self.cracking_patterns = [
            'keygen', 'patch', 'bypass', 'crack', 'patcher', 'key generator'
        ]

        self.anti_tamper_patterns = [
            'vmprotect', 'themida', 'enigma', 'winlicense', 'armadillo',
            'asprotect', 'upx', 'packed', 'obfuscated'
        ]

        self.ransomware_encrypt_patterns = ['encrypt', 'decrypt']
        self.ransomware_context_patterns = ['ransom', 'bitcoin', 'monero', 'recover', 'payment', 'wallet', 'onion', 'tor', 'darknet', 'decryptor']

        self.banking_patterns = [
            'bank', 'paypal', 'visa', 'mastercard', 'amex', 'discover',
            'credit', 'debit', 'card', 'account', 'login', 'password',
            'credentials', 'secure', 'verification'
        ]

    def append_to_buffer(self, text, source=""):
        """Add text to forensic buffer with source header"""
        if source:
            self.buffer.append(f"\n{'═'*60}")
            self.buffer.append(f"📌 {source}")
            self.buffer.append(f"{'═'*60}\n")

        self.buffer.append(text)

        full_text = "\n".join(self.buffer)
        if len(full_text) > self.max_buffer_size:
            self.buffer = [full_text[-self.max_buffer_size:]]

    def get_buffer(self):
        """Get current forensic buffer contents"""
        return "\n".join(self.buffer)

    def clear_buffer(self):
        """Clear the forensic buffer"""
        self.buffer = []

    def build_initial_report(self):
        """Build comprehensive initial report"""
        self._analyze_file_info()
        self._analyze_imports()
        self._analyze_functions()
        self._analyze_strings()
        self._analyze_exports()
        self._analyze_sections()
        self._correlate_behaviors()

        blocks = []
        blocks.append(self._file_summary())
        blocks.append(self._risk_profile())
        blocks.append(self._intent_summary())
        blocks.append(self._correlated_behaviors())
        blocks.append(self._threat_indicators())

        if any(self.iocs.values()):
            ioc_block = ["🌐 IOC SUMMARY:"]
            if self.iocs["urls"]:
                ioc_block.append(f"  URLs: {', '.join(self.iocs['urls'][:5])}")
            if self.iocs["ips"]:
                ioc_block.append(f"  IPs: {', '.join(self.iocs['ips'][:5])}")
            if self.iocs["domains"]:
                ioc_block.append(f"  Domains: {', '.join(self.iocs['domains'][:5])}")
            if self.iocs["paths"]:
                ioc_block.append(f"  Paths: {', '.join(self.iocs['paths'][:5])}")
            blocks.append("\n".join(ioc_block))

        evidence_blocks = []
        if self.raw_function_hits:
            evidence_blocks.append("\n📌 RAW FUNCTION EVIDENCE:")
            evidence_blocks.extend(self.raw_function_hits[:30])
        if self.raw_import_hits:
            evidence_blocks.append("\n📌 RAW IMPORT EVIDENCE:")
            evidence_blocks.extend(self.raw_import_hits[:30])
        if self.raw_string_hits:
            evidence_blocks.append("\n📌 RAW STRING EVIDENCE:")
            evidence_blocks.extend(self.raw_string_hits[:40])
        if self.raw_export_hits:
            evidence_blocks.append("\n📌 RAW EXPORT EVIDENCE:")
            evidence_blocks.extend(self.raw_export_hits[:20])
        if self.raw_section_hits:
            evidence_blocks.append("\n📌 RAW SECTION EVIDENCE:")
            evidence_blocks.extend(self.raw_section_hits[:20])

        if evidence_blocks:
            blocks.append("\n".join(evidence_blocks))

        if self.key_material:
            blocks.append("\n🔑 KEY MATERIAL DETECTED:\n" + "\n".join(self.key_material[:10]))

        if self.protectors_detected:
            blocks.append("\n🛡️ PROTECTORS DETECTED:\n" + "\n".join(set(self.protectors_detected)))

        full = "\n\n".join(blocks)
        self.append_to_buffer(full, "INITIAL FORENSIC ANALYSIS")
        return full

    def _analyze_file_info(self):
        for line in self.analyzer.file_info_data.split("\n"):
            if any(k in line.lower() for k in ["arch", "os", "bits", "subsys", "compiler", "machine"]):
                if ':' in line:
                    self.raw_section_hits.append(f"[FILEINFO] {line.strip()}")

    def _analyze_sections(self):
        for line in self.analyzer.sections_data.split("\n"):
            if any(x in line for x in ['.text', '.data', '.rdata', '.rsrc', '.reloc', 'UPX', 'packed']):
                self.raw_section_hits.append(f"[SECTION] {line.strip()}")

    def _analyze_imports(self):
        for line in self.analyzer.imports_data.split("\n"):
            if "imp." in line or "dll" in line.lower():
                line_lower = line.lower()
                is_interesting = False
                tags = []

                if any(p in line_lower for p in self.network_patterns):
                    self.signal_counts["network"] += 1
                    tags.append("NET")
                    is_interesting = True
                if any(p in line_lower for p in self.crypto_patterns):
                    self.signal_counts["crypto"] += 1
                    tags.append("CRYPTO")
                    is_interesting = True
                if any(p in line_lower for p in self.key_patterns):
                    self.signal_counts["key_material"] += 1
                    tags.append("KEY")
                    is_interesting = True
                if any(p in line_lower for p in self.file_patterns):
                    self.signal_counts["file_ops"] += 1
                    tags.append("FILE")
                    is_interesting = True
                if any(p in line_lower for p in self.process_patterns):
                    self.signal_counts["process_ops"] += 1
                    tags.append("PROC")
                    is_interesting = True
                if any(p in line_lower for p in self.registry_patterns):
                    self.signal_counts["registry_ops"] += 1
                    tags.append("REG")
                    is_interesting = True
                if any(p in line_lower for p in self.licensing_patterns):
                    self.signal_counts["licensing"] += 1
                    tags.append("LIC")
                    is_interesting = True
                if any(p in line_lower for p in self.cracking_patterns):
                    if not any(x in line_lower for x in ['operator', 'destructor', 'vectorcall']):
                        self.signal_counts["cracking"] += 1
                        tags.append("CRACK")
                        is_interesting = True
                if any(p in line_lower for p in self.anti_tamper_patterns):
                    self.signal_counts["anti_tamper"] += 1
                    tags.append("TAMPER")
                    is_interesting = True
                if any(p in line_lower for p in self.ransomware_encrypt_patterns):
                    self.signal_counts["ransomware"] += 1
                    tags.append("ENCRYPT")
                    is_interesting = True
                if any(p in line_lower for p in self.banking_patterns):
                    self.signal_counts["banking"] += 1
                    tags.append("BANK")
                    is_interesting = True

                if is_interesting:
                    tag_str = f"[{'/'.join(tags)}]" if tags else ""
                    self.raw_import_hits.append(f"{tag_str} {line.strip()}")

    def _analyze_functions(self):
        for line in self.analyzer.functions_data.split("\n"):
            if "0x" in line and not line.startswith('==='):
                line_lower = line.lower()
                is_interesting = False
                tags = []

                if any(p in line_lower for p in self.licensing_patterns):
                    self.signal_counts["licensing"] += 1
                    tags.append("LIC")
                    is_interesting = True
                if any(p in line_lower for p in self.cracking_patterns):
                    if not any(x in line_lower for x in ['operator', 'destructor']):
                        self.signal_counts["cracking"] += 1
                        tags.append("CRACK")
                        is_interesting = True
                if any(p in line_lower for p in self.crypto_patterns):
                    self.signal_counts["crypto"] += 1
                    tags.append("CRYPTO")
                    is_interesting = True
                if any(p in line_lower for p in self.network_patterns):
                    self.signal_counts["network"] += 1
                    tags.append("NET")
                    is_interesting = True
                if any(p in line_lower for p in self.file_patterns):
                    self.signal_counts["file_ops"] += 1
                    tags.append("FILE")
                    is_interesting = True
                if any(p in line_lower for p in self.process_patterns):
                    self.signal_counts["process_ops"] += 1
                    tags.append("PROC")
                    is_interesting = True
                if any(p in line_lower for p in ['vmprotect', 'themida', 'enigma', 'winlicense', 'upx']):
                    self.signal_counts["anti_tamper"] += 1
                    tags.append("PROTECTOR")
                    is_interesting = True

                if is_interesting:
                    tag_str = f"[{'/'.join(tags)}]" if tags else ""
                    self.raw_function_hits.append(f"{tag_str} {line.strip()}")

    def _analyze_strings(self):
        url_pattern = r'https?://[^\s"\']+|ftp://[^\s"\']+'
        ip_pattern = r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'
        domain_pattern = r'\b[a-zA-Z0-9][a-zA-Z0-9\-]{0,61}[a-zA-Z0-9]?\.[a-zA-Z]{2,}\b'
        hex_key_pattern = r'[A-Fa-f0-9]{32,}'
        base64_key_pattern = r'[A-Za-z0-9+/]{40,}='
        pem_pattern = r'-----BEGIN [A-Z ]+-----'

        for line in self.analyzer.strings_data.split("\n"):
            if "ascii" in line:
                parts = line.split("ascii")
                if len(parts) > 1:
                    s = parts[-1].strip()
                    is_interesting = False
                    tags = []

                    if re.search(url_pattern, s, re.I):
                        self.signal_counts["network"] += 1
                        tags.append("URL")
                        is_interesting = True
                        self.iocs["urls"].append(s[:100])
                    elif re.search(ip_pattern, s):
                        self.signal_counts["network"] += 1
                        tags.append("IP")
                        is_interesting = True
                        self.iocs["ips"].append(s)
                    elif re.search(domain_pattern, s) and not re.search(r'http|https|ftp', s):
                        if len(s) > 6 and '.' in s:
                            self.signal_counts["network"] += 1
                            tags.append("DOMAIN")
                            is_interesting = True
                            self.iocs["domains"].append(s)
                    elif looks_like_real_path(s):
                        self.signal_counts["file_ops"] += 1
                        tags.append("PATH")
                        is_interesting = True
                        self.iocs["paths"].append(s[:80])

                    if re.fullmatch(hex_key_pattern, s):
                        self.signal_counts["key_material"] += 1
                        tags.append("HEXKEY")
                        is_interesting = True
                        if len(s) == 32:
                            self.key_material.append("AES-128 key (32 hex chars)")
                        elif len(s) == 64:
                            self.key_material.append("AES-256 key (64 hex chars)")
                    elif re.fullmatch(base64_key_pattern, s):
                        self.signal_counts["key_material"] += 1
                        tags.append("B64KEY")
                        is_interesting = True
                        self.key_material.append("Base64-encoded key/cert")
                    elif re.search(pem_pattern, s, re.I):
                        self.signal_counts["key_material"] += 1
                        tags.append("PEM")
                        is_interesting = True
                        self.key_material.append("PEM formatted key/cert")

                    s_lower = s.lower()
                    if any(p in s_lower for p in self.licensing_patterns):
                        self.signal_counts["licensing"] += 1
                        tags.append("LIC")
                        is_interesting = True
                    if any(p in s_lower for p in self.cracking_patterns):
                        if not any(x in s_lower for x in ['operator', 'destructor', 'vectorcall']):
                            self.signal_counts["cracking"] += 1
                            tags.append("CRACK")
                            is_interesting = True

                    has_encrypt = any(p in s_lower for p in self.ransomware_encrypt_patterns)
                    has_ransom_context = any(p in s_lower for p in self.ransomware_context_patterns)
                    if has_encrypt and has_ransom_context:
                        self.signal_counts["ransomware"] += 2
                        tags.append("RANSOM")
                        is_interesting = True

                    if any(p in s_lower for p in self.banking_patterns):
                        self.signal_counts["banking"] += 1
                        tags.append("BANK")
                        is_interesting = True

                    if any(p in s_lower for p in self.anti_tamper_patterns):
                        self.signal_counts["anti_tamper"] += 1
                        tags.append("TAMPER")
                        is_interesting = True
                        if 'vmprotect' in s_lower:
                            self.protectors_detected.append("VMProtect")
                        if 'themida' in s_lower:
                            self.protectors_detected.append("Themida")
                        if 'enigma' in s_lower:
                            self.protectors_detected.append("Enigma Protector")
                        if 'upx' in s_lower:
                            self.protectors_detected.append("UPX Packer")

                    if is_interesting and looks_like_readable_string(s):
                        tag_str = f"[{'/'.join(tags)}]" if tags else ""
                        self.raw_string_hits.append(f"{tag_str} {line.strip()}")
                        self.raw_string_hits.append(f"   → {s[:100]}")

    def _analyze_exports(self):
        if not self.analyzer.exports_data:
            return

        for line in self.analyzer.exports_data.split("\n"):
            if "0x" in line and line.strip():
                line_lower = line.lower()
                tags = []
                if any(p in line_lower for p in self.crypto_patterns):
                    tags.append("CRYPTO")
                if any(p in line_lower for p in self.network_patterns):
                    tags.append("NET")
                tag_str = f"[{'/'.join(tags)}]" if tags else ""
                self.raw_export_hits.append(f"{tag_str} [EXPORT] {line.strip()}")

    def _correlate_behaviors(self):
        if self.signal_counts["crypto"] > 0 and self.signal_counts["file_ops"] > 2 and self.signal_counts["ransomware"] > 0:
            self.correlated_behaviors.append("🔴 HIGH: File encryption + ransomware context detected")
            self.signal_counts["ransomware"] += 3
        if self.signal_counts["network"] > 1 and self.signal_counts["crypto"] > 1:
            self.correlated_behaviors.append("🟡 MEDIUM: Encrypted network communication possible")
        if self.signal_counts["process_ops"] > 2 and "virtualalloc" in self.analyzer.imports_data.lower():
            self.correlated_behaviors.append("🔴 HIGH: Process injection capabilities detected")
        if self.signal_counts["registry_ops"] > 0 and self.signal_counts["persistence"] > 0:
            self.correlated_behaviors.append("🟡 MEDIUM: Registry-based persistence mechanism")
        if self.signal_counts["anti_debug"] > 1:
            self.correlated_behaviors.append("🔴 HIGH: Anti-debug/anti-analysis techniques present")
        if self.signal_counts["cracking"] > 1:
            self.correlated_behaviors.append("🔴 HIGH: Software cracking/piracy related content")
        if self.signal_counts["licensing"] > 1 and self.signal_counts["cracking"] > 0:
            self.correlated_behaviors.append("🔴 HIGH: License validation bypass attempts")
        elif self.signal_counts["licensing"] > 1:
            self.correlated_behaviors.append("🟡 MEDIUM: License/DRM validation logic present")
        if self.signal_counts["anti_tamper"] > 0:
            self.correlated_behaviors.append("🔴 HIGH: Anti-tampering/obfuscation detected")
        if self.signal_counts["crypto"] > 1 and self.signal_counts["cracking"] > 0:
            self.correlated_behaviors.append("🔴 HIGH: Possible keygen/patch generator")
        if self.signal_counts["banking"] > 1 or (self.signal_counts["key_material"] > 1 and "password" in self.analyzer.strings_data.lower()):
            self.correlated_behaviors.append("🔴 HIGH: Possible credential theft capabilities")
        if self.signal_counts["network"] > 2 and any(p in self.analyzer.imports_data.lower() for p in ['url', 'download', 'http']):
            self.correlated_behaviors.append("🟡 MEDIUM: Possible downloader/loader functionality")

    def _file_summary(self):
        lines = []
        for line in self.analyzer.file_info_data.split("\n"):
            if any(k in line.lower() for k in ["arch", "os", "bits", "subsys", "compiler", "machine"]):
                if ':' in line:
                    lines.append(line.strip())
        if self.analyzer.entry_point:
            lines.append(f"Entry Point: {self.analyzer.entry_point}")
        return "📋 FILE INFO:\n" + "\n".join(lines[:8])

    def _risk_profile(self):
        lines = []
        total_risk = 0
        for category, count in self.signal_counts.items():
            if count > 0:
                score = count * self.weights[category]
                total_risk += score
                indicator = "🔴" if score > 10 else "🟡" if score > 3 else "🟢"
                lines.append(f"{indicator} {category.upper()}: {score} (hits: {count})")

        if total_risk > 50:
            risk_level = "🔴 CRITICAL"
        elif total_risk > 20:
            risk_level = "🟡 HIGH"
        elif total_risk > 5:
            risk_level = "🟢 MEDIUM"
        else:
            risk_level = "⚪ LOW"

        return f"🎯 RISK PROFILE ({risk_level} - total: {total_risk}):\n" + "\n".join(lines[:10])

    def _intent_summary(self):
        intents = []
        if self.signal_counts["network"] > 1:
            intents.append("🌐 Network communication capabilities")
        if self.signal_counts["crypto"] > 0:
            intents.append("🔐 Cryptographic functions present")
        if self.signal_counts["file_ops"] > 2:
            intents.append("📁 File system access")
        if self.signal_counts["process_ops"] > 1:
            intents.append("⚙️ Process manipulation capabilities")
        if self.signal_counts["registry_ops"] > 0:
            intents.append("📝 Registry interaction")
        if self.signal_counts["ransomware"] > 0:
            intents.append("⚠️ Ransomware-related strings detected")
        if self.signal_counts["banking"] > 0:
            intents.append("💰 Banking/financial strings present")
        if self.signal_counts["licensing"] > 0:
            intents.append("🔒 License/DRM validation logic")
        if self.signal_counts["cracking"] > 0:
            intents.append("🔓 Cracking/piracy related content")
        if self.signal_counts["anti_tamper"] > 0:
            intents.append("🛡️ Anti-tampering protections")
        if not intents:
            intents.append("❓ No clear behavior patterns detected")
        return "🎯 INTENT SUMMARY:\n" + "\n".join(intents[:8])

    def _correlated_behaviors(self):
        if not self.correlated_behaviors:
            return "🔗 CORRELATED BEHAVIORS:\nNone detected"
        return "🔗 CORRELATED BEHAVIORS:\n" + "\n".join(self.correlated_behaviors[:6])

    def _threat_indicators(self):
        indicators = []
        if self.signal_counts["ransomware"] > 1:
            indicators.append("☣️ RANSOMWARE: Encryption + ransom context detected")
        elif self.signal_counts["ransomware"] > 0:
            indicators.append("⚠️ Ransomware-related strings present")
        if self.signal_counts["banking"] > 1:
            indicators.append("💰 Banking trojan indicators present")
        if self.signal_counts["cracking"] > 1:
            indicators.append("🔓 Cracking/piracy tool indicators")
        if self.signal_counts["licensing"] > 1:
            indicators.append("🔒 License/DRM validation logic")
        if self.signal_counts["anti_tamper"] > 0:
            indicators.append("🛡️ Anti-tampering/obfuscation detected")
        if not indicators:
            return "🚨 THREAT INDICATORS:\nNone specific"
        return "🚨 THREAT INDICATORS:\n" + "\n".join(indicators[:5])

    def get_relevant_counts(self):
        return {
            'functions': len(set(self.raw_function_hits)),
            'strings': len(set(self.raw_string_hits)),
            'imports': len(set(self.raw_import_hits)),
            'exports': len(set(self.raw_export_hits))
        }

# ============================================================================
# Real Radare2 analyzer
# ============================================================================
class LiveRadare2Analyzer:
    def __init__(self, binary_path):
        self.binary_path = binary_path
        self.process = None
        self.analysis_cache = {}
        self.last_command_output = ""
        self.analysis_complete = False
        self.file_type = None
        self.entry_point = None
        self.exports_data = ""

        self.functions_data = ""
        self.strings_data = ""
        self.imports_data = ""
        self.sections_data = ""
        self.file_info_data = ""

        self.functions_loaded = False
        self.strings_loaded = False
        self.imports_loaded = False
        self.sections_loaded = False
        self.file_info_loaded = False
        self.exports_loaded = False

        self.forensic = None

        if binary_path:
            self.file_type = FileTypeDetector.detect(os.path.basename(binary_path))
            print(f"📌 Detected: {self.file_type['description']}")

    def execute_command(self, cmd, timeout=15):
        try:
            flags = " ".join(self.file_type['r2_flags']) if self.file_type and self.file_type['r2_flags'] else ""

            if flags:
                full_cmd = f"r2 {flags} -q -c '{cmd}' {self.binary_path}"
            else:
                full_cmd = f"r2 -q -c '{cmd}' {self.binary_path}"

            result = subprocess.run(
                full_cmd,
                shell=True,
                capture_output=True,
                text=True,
                timeout=timeout
            )

            output = result.stdout if result.stdout else result.stderr
            self.last_command_output = output
            return output if output else "(No output)"

        except subprocess.TimeoutExpired:
            return "⚠️ Command execution timed out"
        except Exception as e:
            return f"❌ Execution error: {str(e)}"

    def auto_load_all(self):
        results = {}

        self.file_info_data = self.execute_command("iI")
        self.file_info_loaded = True
        results['file_info'] = self.file_info_data

        self.execute_command("aaaa")

        self.functions_data = self.execute_command("afl")

        if self.file_type and self.file_type['is_dll']:
            exports = self.execute_command("iE")
            if exports and len(exports.strip()) > 10:
                self.exports_data = exports
                self.exports_loaded = True

        if not self.functions_data or len(self.functions_data.strip()) < 20:
            self.functions_data = self.execute_command("is")

        self.functions_loaded = True
        results['functions'] = self.functions_data

        self.strings_data = self.execute_command("izz")
        if not self.strings_data or len(self.strings_data.split('\n')) < 5:
            self.strings_data = self.execute_command("iz")
        self.strings_loaded = True
        results['strings'] = self.strings_data

        self.imports_data = self.execute_command("ii")
        self.imports_loaded = True
        results['imports'] = self.imports_data

        self.sections_data = self.execute_command("iS")
        self.sections_loaded = True
        results['sections'] = self.sections_data

        entry = self.execute_command("ie")
        match = re.search(r'entry0\s+(0x[0-9a-f]+)', entry)
        if not match:
            match = re.search(r'0x[0-9a-f]+', entry)
        if match:
            self.entry_point = match.group(0)

        if not self.entry_point:
            funcs = self.functions_data
            match = re.search(r'(0x[0-9a-f]+)', funcs)
            if match:
                self.entry_point = match.group(1)

        self.analysis_complete = True
        results['entry'] = self.entry_point or "Not found"

        self.forensic = ForensicBufferClassifier(self)
        self.forensic.build_initial_report()

        return results

    def get_forensic_buffer(self):
        if not self.forensic:
            return "No forensic data available"
        return self.forensic.get_buffer()

    def append_command_to_buffer(self, cmd, output):
        if not self.forensic:
            return "No forensic buffer initialized"
        header = f"COMMAND: {cmd}"
        self.forensic.append_to_buffer(output, header)
        return self.forensic.get_buffer()

    def clear_forensic_buffer(self):
        if self.forensic:
            self.forensic.clear_buffer()
            return "Forensic buffer cleared"
        return "No forensic buffer"

    def get_relevant_counts(self):
        if not self.forensic:
            return {'functions': 0, 'strings': 0, 'imports': 0, 'exports': 0}
        return self.forensic.get_relevant_counts()

    def disassemble_function(self, function_name):
        if function_name.startswith('0x'):
            return self.execute_command(f"pdf @ {function_name}")

        sym_check = self.execute_command(f"is~{function_name}")
        if sym_check and function_name in sym_check:
            match = re.search(r'0x[0-9a-f]+', sym_check)
            if match:
                return self.execute_command(f"pdf @ {match.group(0)}")

        if self.file_type and self.file_type['is_dll']:
            exports = self.execute_command(f"iE~{function_name}")
            if exports and function_name in exports:
                match = re.search(r'0x[0-9a-f]+', exports)
                if match:
                    return self.execute_command(f"pdf @ {match.group(0)}")

        result = self.execute_command(f"pdf @ {function_name}")

        if "Invalid" in result and self.entry_point:
            return self.execute_command(f"pd 20 @ {self.entry_point}")

        return result

# ============================================================================
# Live Analysis Session
# ============================================================================
class LiveAnalysisSession:
    def __init__(self):
        self.analyzer = None
        self.binary_path = None
        self.binary_name = None
        self.llm = LocalLLMAssistant("Qwen/Qwen2.5-0.5B")

    def load_and_analyze(self, file_obj):
        if file_obj is None:
            return "❌ No file uploaded", "", "", "", "", "", "❌ No file", [], ""

        try:
            if hasattr(file_obj, 'name'):
                file_path = file_obj.name
                self.binary_name = os.path.basename(file_path)

                with open(file_path, 'rb') as src:
                    with tempfile.NamedTemporaryFile(delete=False, suffix=os.path.splitext(self.binary_name)[1]) as dst:
                        dst.write(src.read())
                        self.binary_path = dst.name
            else:
                self.binary_name = getattr(file_obj, 'name', 'uploaded_file.bin')
                content = file_obj.read()
                with tempfile.NamedTemporaryFile(delete=False, suffix=os.path.splitext(self.binary_name)[1]) as tmp:
                    tmp.write(content)
                    self.binary_path = tmp.name

            self.analyzer = LiveRadare2Analyzer(self.binary_path)
            results = self.analyzer.auto_load_all()
            forensic_buffer = self.analyzer.get_forensic_buffer()

            # Set context for LLM
            self.llm.set_context(forensic_buffer)

            relevant = self.analyzer.get_relevant_counts()
            status = f"✅ Loaded: {relevant['functions']} relevant functions, {relevant['strings']} relevant strings, {relevant['imports']} relevant imports"

            commands = self.get_enhanced_commands()
            cmd_choices = [f"{cmd[0]}|{cmd[1]}" for cmd in commands]

            return (status,
                    results['file_info'],
                    results['functions'],
                    results['strings'],
                    results['imports'],
                    results['sections'],
                    f"Type: {self.analyzer.file_type['description']}",
                    cmd_choices,
                    forensic_buffer)

        except Exception as e:
            return f"❌ Error: {str(e)}", "", "", "", "", "", "❌ Failed", [], f"Error: {str(e)}"

    def get_enhanced_commands(self):
        if self.analyzer and self.analyzer.file_type and self.analyzer.file_type['is_dll']:
            return [
                ("iE", "📤 List DLL exports"),
                ("ii", "📥 List DLL imports"),
                ("iS", "📐 List sections"),
                ("iz", "🔣 Show strings"),
                ("afl", "📋 List all functions"),
                ("pdf @ entry0", "📄 Disassemble entry point"),
                ("axt @ entry0", "🔎 Find references"),
                ("iD", "💾 Data references"),
                ("/v 55 8b ec", "🔍 Search bytes"),
                ("ood", "🐛 Debug mode")
            ]
        else:
            return [
                ("pdf @ entry0", "📄 Disassemble entry point"),
                ("afl", "📋 List functions"),
                ("ii", "📦 Show imports"),
                ("iS", "📐 Show sections"),
                ("iz", "🔣 Show strings"),
                ("iE", "🎯 Show exports"),
                ("axt @ entry0", "🔎 Find references"),
                ("iD", "💾 Data references"),
                ("/v 55 8b ec", "🔍 Search bytes"),
                ("ood", "🐛 Debug mode")
            ]

    def execute_command(self, cmd):
        if not self.analyzer:
            return "Load a binary first", ""

        if cmd and '|' in cmd:
            cmd = cmd.split('|')[0]

        if cmd.startswith("pdf @") or cmd.startswith("af @"):
            parts = cmd.split('@')
            if len(parts) > 1:
                func = parts[1].strip()
                output = self.analyzer.disassemble_function(func)
            else:
                output = self.analyzer.execute_command(cmd)
        else:
            output = self.analyzer.execute_command(cmd)

        updated_buffer = self.analyzer.append_command_to_buffer(cmd, output)
        return output, updated_buffer

    def clear_buffer(self):
        if not self.analyzer:
            return "No analyzer loaded"
        return self.analyzer.clear_forensic_buffer()

    def chat_with_llm(self, user_input, history):
        if not self.analyzer or not self.analyzer.analysis_complete:
            return history + [(user_input, "⚠️ Analysis in progress...")]

        forensic_buffer = self.analyzer.get_forensic_buffer()
        llm_response = self.llm.query(user_input, forensic_buffer)

        history = history + [(user_input, llm_response)]
        return history

# ============================================================================
# Custom CSS for Vibrant Hacker Theme
# ============================================================================
custom_css = """
/* Global dark theme - NO BLUR */
.gradio-container {
    background-color: #0a0c14 !important;
}

/* Main container */
.container {
    max-width: 100% !important;
    background: #0c0f17 !important;
}

/* Main chartreuse color for primary text */
body, .gr-box, .gr-textbox, .gr-markdown {
    color: #ccff00 !important;  /* Vibrant chartreuse */
}

/* Headers with specific colors */
h1 {
    font-family: 'JetBrains Mono', 'Fira Code', monospace !important;
    color: #ccff00 !important;  /* Chartreuse */
    text-shadow: 0 0 8px #ccff00 !important;
    border-bottom: 2px solid #ccff00;
    padding-bottom: 10px;
}

/* File Info - Hot Pink */
.gr-box:has(.markdown:contains("File Info")) .markdown h3 {
    color: #ff44cc !important;  /* Vibrant pink */
    text-shadow: 0 0 5px #ff44cc !important;
}
.gr-box:has(.markdown:contains("File Info")) textarea {
    border-left: 4px solid #ff44cc !important;
    color: #ff88cc !important;
}

/* Functions/Exports - Electric Blue */
.gr-box:has(.markdown:contains("Functions")) .markdown h3 {
    color: #44ccff !important;  /* Electric blue */
    text-shadow: 0 0 5px #44ccff !important;
}
.gr-box:has(.markdown:contains("Functions")) textarea {
    border-left: 4px solid #44ccff !important;
    color: #88ddff !important;
}

/* Strings - Chartreuse */
.gr-box:has(.markdown:contains("Strings")) .markdown h3 {
    color: #ccff00 !important;  /* Chartreuse */
    text-shadow: 0 0 5px #ccff00 !important;
}
.gr-box:has(.markdown:contains("Strings")) textarea {
    border-left: 4px solid #ccff00 !important;
    color: #ddff88 !important;
}

/* Imports - Orange */
.gr-box:has(.markdown:contains("Imports")) .markdown h3 {
    color: #ffaa00 !important;  /* Orange */
    text-shadow: 0 0 5px #ffaa00 !important;
}
.gr-box:has(.markdown:contains("Imports")) textarea {
    border-left: 4px solid #ffaa00 !important;
    color: #ffcc66 !important;
}

/* Sections - Purple */
.gr-box:has(.markdown:contains("Sections")) .markdown h3 {
    color: #aa66ff !important;  /* Purple */
    text-shadow: 0 0 5px #aa66ff !important;
}
.gr-box:has(.markdown:contains("Sections")) textarea {
    border-left: 4px solid #aa66ff !important;
    color: #cc99ff !important;
}

/* Forensic Buffer - Magenta */
.gr-box:has(.label:contains("FORENSIC BUFFER")) .markdown h3 {
    color: #ff44aa !important;  /* Magenta */
    text-shadow: 0 0 5px #ff44aa !important;
}
.gr-box:has(.label:contains("FORENSIC BUFFER")) textarea {
    border-left: 4px solid #ff44aa !important;
    color: #ffaadd !important;
}

/* Command Output - Cyan */
.gr-box:has(.label:contains("Command Output")) .label-text {
    color: #44ffcc !important;
}
.gr-box:has(.label:contains("Command Output")) textarea {
    border-left: 4px solid #44ffcc !important;
    color: #88ffee !important;
}

/* Buttons - No blur, solid vibrant colors */
button.primary {
    background: #ccff00 !important;
    color: #0a0c14 !important;
    font-weight: bold !important;
    border: 2px solid #ffffff !important;
}

button.primary:hover {
    background: #aadd00 !important;
    transform: translateY(-2px) !important;
}

button.secondary {
    background: transparent !important;
    border: 2px solid #44ccff !important;
    color: #44ccff !important;
}

button.secondary:hover {
    background: #44ccff !important;
    color: #0a0c14 !important;
}

/* Execute button */
button:has(.label:contains("Execute")) {
    background: #ff44cc !important;
    color: #0a0c14 !important;
    border: 2px solid #ffffff !important;
}

/* Tabs with different colors */
.tabs > .tab-nav button {
    font-family: 'JetBrains Mono', monospace !important;
    font-weight: bold !important;
    text-transform: uppercase !important;
}

.tabs > .tab-nav button:nth-child(1) {
    color: #44ccff !important;  /* Blue */
}
.tabs > .tab-nav button:nth-child(2) {
    color: #ff44aa !important;  /* Pink */
}
.tabs > .tab-nav button:nth-child(3) {
    color: #ccff00 !important;  /* Chartreuse */
}

.tabs > .tab-nav button.selected {
    border-bottom: 3px solid currentColor !important;
    font-weight: bold !important;
}

/* Dropdowns */
select, .gr-dropdown {
    background: #0f1322 !important;
    color: #ccff00 !important;
    border: 2px solid #44ccff !important;
}

/* Labels */
label, .label-text {
    color: #ccff00 !important;
    font-weight: bold !important;
    text-transform: uppercase !important;
}

/* File upload area */
.gr-file {
    border: 2px solid #ff44cc !important;
    background: rgba(255, 68, 204, 0.1) !important;
}

/* Chat area */
.gr-box:has(.label:contains("Chat")) textarea {
    border-left: 4px solid #aa66ff !important;
    color: #cc99ff !important;
}

/* Scrollbar - solid colors */
::-webkit-scrollbar {
    width: 12px;
    background: #0a0c14;
}

::-webkit-scrollbar-thumb {
    background: #ccff00;
    border: 2px solid #0a0c14;
}

::-webkit-scrollbar-thumb:hover {
    background: #44ccff;
}

/* Status box */
.gr-box.status {
    background: #0f1322 !important;
    border-left: 4px solid #ccff00 !important;
    color: #ccff00 !important;
}
"""

# ============================================================================
# Gradio Interface with Custom Theme
# ============================================================================
def create_live_interface():
    session = LiveAnalysisSession()

    with gr.Blocks(title="🔴 LIVE Binary Analysis - HACKER EDITION", css=custom_css) as demo:
        gr.Markdown("# 🔴 LIVE Binary Analysis with Radare2 + Forensic Classifier + AI (Qwen 0.5B)")
        gr.Markdown("**HACKER EDITION** | Static analysis with evidence collection | AI connected to forensic buffer")

        # File upload
        with gr.Row():
            file_input = gr.File(label="📁 Upload Binary", file_types=None)
            load_btn = gr.Button("🚀 Upload & Analyze", variant="primary", size="lg")
            upload_status = gr.Textbox(label="Status", value="Ready", interactive=False, lines=1)

        # Main tabs
        with gr.Tabs():
            with gr.TabItem("📊 Raw Analysis Data"):
                with gr.Row():
                    with gr.Column():
                        gr.Markdown("### 📋 File Info")
                        info_output = gr.Textbox(label="", lines=6, interactive=False)
                    with gr.Column():
                        gr.Markdown("### 📋 Functions/Exports")
                        func_output = gr.Textbox(label="", lines=6, interactive=False)

                with gr.Row():
                    with gr.Column():
                        gr.Markdown("### 🔣 Strings")
                        strings_output = gr.Textbox(label="", lines=6, interactive=False)
                    with gr.Column():
                        gr.Markdown("### 📦 Imports")
                        imports_output = gr.Textbox(label="", lines=6, interactive=False)

                with gr.Row():
                    with gr.Column():
                        gr.Markdown("### 📐 Sections")
                        sections_output = gr.Textbox(label="", lines=4, interactive=False)
                    with gr.Column():
                        analysis_status = gr.Textbox(label="Type", lines=2, interactive=False)

            with gr.TabItem("🧠 FORENSIC BUFFER"):
                forensic_output = gr.Textbox(label="Complete Forensic Analysis + Evidence", lines=30, interactive=False)
                with gr.Row():
                    clear_buffer_btn = gr.Button("🗑️ Clear Buffer", variant="secondary")
                    refresh_buffer_btn = gr.Button("🔄 Refresh", variant="secondary")

            with gr.TabItem("💬 AI Assistant"):
                chatbot = gr.Chatbot(label="Conversation with AI (Qwen 0.5B)", height=400)
                with gr.Row():
                    msg = gr.Textbox(label="Question about the binary", placeholder="What does the forensic buffer show?", scale=4)
                    send_btn = gr.Button("Send", variant="primary", scale=1)
                    clear_chat_btn = gr.Button("Clear Chat", variant="secondary")

        # Commands
        with gr.Row():
            with gr.Column(scale=2):
                gr.Markdown("### 📋 Commands (Append to Forensic Buffer)")
                cmd_dropdown = gr.Dropdown(
                    choices=[],
                    label="Quick Commands"
                )
                cmd_input = gr.Textbox(label="Custom Command", placeholder="pdf @ entry0", lines=2)
                with gr.Row():
                    exec_btn = gr.Button("⚡ Execute & Append", variant="primary", scale=2)
                    cmd_output = gr.Textbox(label="Command Output", lines=4, scale=3)

        # Event handlers
        def load_and_analyze(file):
            if file is None:
                return ["❌ No file"] + [""] * 5 + ["No file"] + [[]] + [""]

            status, info, funcs, strings, imports, sections, file_type, cmds, forensic = session.load_and_analyze(file)

            return [status, info, funcs, strings, imports, sections, file_type,
                   gr.Dropdown(choices=cmds), forensic]

        def execute_custom(cmd):
            output, updated_buffer = session.execute_command(cmd)
            return output, updated_buffer

        def update_cmd_input(choice):
            return choice.split('|')[0] if choice and '|' in choice else choice

        def clear_buffer():
            session.clear_buffer()
            if session.analyzer:
                return session.analyzer.get_forensic_buffer()
            return "Buffer cleared"

        def refresh_buffer():
            if session.analyzer:
                return session.analyzer.get_forensic_buffer()
            return "No analyzer loaded"

        def chat_message(message, history):
            return session.chat_with_llm(message, history)

        # Wire up
        load_btn.click(
            load_and_analyze,
            inputs=[file_input],
            outputs=[upload_status, info_output, func_output, strings_output,
                    imports_output, sections_output, analysis_status,
                    cmd_dropdown, forensic_output]
        )

        cmd_dropdown.change(
            update_cmd_input,
            inputs=[cmd_dropdown],
            outputs=[cmd_input]
        )

        exec_btn.click(
            execute_custom,
            inputs=[cmd_input],
            outputs=[cmd_output, forensic_output]
        )

        clear_buffer_btn.click(
            clear_buffer,
            inputs=[],
            outputs=[forensic_output]
        )

        refresh_buffer_btn.click(
            refresh_buffer,
            inputs=[],
            outputs=[forensic_output]
        )

        send_btn.click(
            chat_message,
            inputs=[msg, chatbot],
            outputs=[chatbot]
        ).then(lambda: "", None, [msg])

        msg.submit(
            chat_message,
            inputs=[msg, chatbot],
            outputs=[chatbot]
        ).then(lambda: "", None, [msg])

        clear_chat_btn.click(lambda: [], None, chatbot)

    return demo

# Launch
print("🚀 Launching FORENSIC BUFFER binary analysis with Qwen 0.5B AI...")
print("✅ Qwen 2.5 0.5B AI Assistant (tokenless)")

demo = create_live_interface()
demo.launch(share=True, debug=False)